In [ ]:
import requests


url = "https://api.divar.ir/v8/postlist/w/search"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36',
    'Content-Type': 'application/json',
    'Referer': 'https://divar.ir/'
}

payload = {
    "city_ids": ["1"], 
    "search_data": {
        "form_data": {
            "data": {
                "category": {"str": {"value": "real-estate"}}
            }
        }
    }
}

print("Sending request to Divar...")
response = requests.post(url, headers=headers, json=payload)

if response.status_code == 200:
    print("Success! Status 200")
    data = response.json()
    
    print("Keys in response:", data.keys())
    
    if 'pagination' in data:
        print("\nNext page info:", data['pagination'])
        
else:
    print("Error:", response.status_code)
    print(response.text)


In [ ]:
import requests
import pandas as pd
import os
import time
import random


search_url = "https://api.divar.ir/v8/postlist/w/search"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Content-Type": "application/json"
}

all_ads = []
last_post_date = ""
page = 1
max_pages = 450

while page <= max_pages:
    print(f"Page : {page}...")
    
    payload = {
        "city_ids": ["1"],
        "search_data": {
            "form_data": {
                "data": {
                    "category": {"str": {"value": "residential-sell"}}
                }
            }
        }
    }
    
    if last_post_date:
        payload["pagination_data"] = {
            "@type": "type.googleapis.com/post_list.PaginationData",
            "last_post_date": last_post_date,
            "page": page,
            "layer_page": page
        }

    try:
        response = requests.post(search_url, json=payload, headers=headers)
        if response.status_code == 200:
            data = response.json()
            widgets = data.get("list_widgets", [])
            
            for widget in widgets:
                if widget.get("widget_type") == "POST_ROW":
                    post_data = widget.get("data", {})
                    token = post_data.get("token")
                    district = post_data.get("action", {}).get("payload", {}).get("web_info", {}).get("district_persian", "Nan")
                    
                    if token:
                        all_ads.append({
                            "token": token,
                            "district": district
                        })
            
            pagination = data.get("pagination", {})
            last_post_date = pagination.get("data", {}).get("last_post_date")
            
            if page % 50 == 0:
                pd.DataFrame(all_ads).to_csv("divar_tokens_backup.csv", index=False)
                print(f"--- Backup saved at page {page} ---")

            
            if not last_post_date:
                break
            page += 1
            time.sleep(random.uniform(1,4))
        else:
            break
    except Exception as e:
        print(f"Phase 1 Error: {e}. Retrying in 10 seconds...")
        time.sleep(10)

